# Data-engineering questions

## Q1. Which circuits convert a front-row start into a podium most often, and does that ranking change after the 2014 hybrid era?

**Data used:** the *cleaned* tables from the cleaning section (`c["results"]`, `c["races"]`, `c["circuits"]`, `c["qualifying"]`), not the raw CSVs, so the answer rests on the same cleaning decisions as the rest of the pipeline (sentinels → NaN, dtypes fixed, one row per `(raceId, driverId)`, non-starters removed).

**Tables joined (multi-hop):** `results` → `races` (year, circuitId) → `circuits` (name) and `results` ⟕ `qualifying` on (`raceId`, `driverId`).
`results.csv` alone cannot answer this: it has no circuit and no qualifying position.

**Definitions**
- *Front-row start (primary)*: `results.grid ∈ {1, 2}`, the slot the driver actually started from, after penalties.
- *Front-row qualifier (sensitivity)*: `qualifying.position ∈ {1, 2}`, the on-track qualifying result before penalties.
- *Podium*: `results.positionOrder ≤ 3`.
- *Eras*: pre-hybrid = every season in the dataset up to 2013 (from 1950), hybrid = 2014 onwards. The **full dataset** is used, with no year cut.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr

assert "c" in globals() and all(t in c for t in ["results", "races", "circuits", "qualifying"]),     "Run the cleaning section first: this section reads the cleaned tables in `c`."

HYBRID_START = 2014   # first hybrid-era season
MIN_N        = 10     # min front-row starts per circuit per era to be ranked (about 5 races)

## 1. Take the columns Q1 needs from the cleaned tables

In [ ]:
q1_results = c["results"][["raceId", "driverId", "grid", "positionOrder"]].copy()
q1_races   = c["races"][["raceId", "year", "circuitId"]].copy()
q1_circ    = c["circuits"][["circuitId", "name", "country"]].rename(columns={"name": "circuit_name"})
q1_qual    = c["qualifying"][["raceId", "driverId", "position"]].rename(columns={"position": "quali_position"})

for n, q1_d in [("results", q1_results), ("races", q1_races), ("circuits", q1_circ), ("qualifying", q1_qual)]:
    print(f"{n:11s}", q1_d.shape)

## 2. Grain and joins

Grain: **one row per (`raceId`, `driverId`)**. The cleaning section already enforced this on `c["results"]` (Rule A dedup, then removal of non-starters), so here we only *verify* it and check that each join keeps it. Qualifying must also be unique on the key for a 1:1 join.

In [ ]:
key = ["raceId", "driverId"]
assert not q1_results.duplicated(key).any(), "cleaned results is not unique on (raceId, driverId)"
print("duplicated (raceId, driverId) in cleaned qualifying:", q1_qual.duplicated(key).sum())
q1_qual = q1_qual.drop_duplicates(key, keep="first")

q1_df = (q1_results.merge(q1_races, on="raceId", how="left", validate="m:1")
                .merge(q1_circ, on="circuitId", how="left", validate="m:1")
                .merge(q1_qual, on=key, how="left", validate="1:1"))

assert q1_df["year"].notna().all() and q1_df["circuit_name"].notna().all(), "a join lost rows"
assert not q1_df.duplicated(key).any()
assert len(q1_df) == len(q1_results), "joins changed the number of rows"

# plain numeric dtypes (cleaned tables use nullable Int64)
for col in ["grid", "positionOrder", "quali_position"]:
    q1_df[col] = q1_df[col].astype("float64")
q1_df["year"] = q1_df["year"].astype(int)

FIRST_YEAR, END_YEAR = int(q1_df["year"].min()), int(q1_df["year"].max())
ERA_LABEL = {"pre": f"{FIRST_YEAR}–{HYBRID_START-1}", "post": f"{HYBRID_START}–{END_YEAR}"}
q1_df["era"] = np.where(q1_df["year"] >= HYBRID_START, "post", "pre")

q1_df["podium"]      = (q1_df["positionOrder"] <= 3).astype(int)
q1_df["front_grid"]  = q1_df["grid"].isin([1, 2])            # actual starting slot (after penalties)
q1_df["front_quali"] = q1_df["quali_position"].isin([1, 2])  # on-track qualifying result

print("analysis window (full cleaned dataset):", ERA_LABEL)
print("entries:", len(q1_df), "| races:", q1_df["raceId"].nunique(), "| circuits:", q1_df["circuitId"].nunique())

## 3. Missingness checks

Three things can silently distort the numbers: (a) drivers with **no qualifying row** (qualifying data does not exist for the early decades, so the qualifying-based sensitivity check only covers seasons where it does), (b) **`grid = 0`** (pit-lane start or no grid slot; these are *not* front row), (c) circuits with **too few front-row starts** in an era to rank. Each is quantified below.

In [ ]:
miss = (q1_df.groupby("era")
          .agg(entries=("podium", "size"),
               quali_missing_pct=("quali_position", lambda s: 100 * s.isna().mean()),
               grid0_pct=("grid", lambda s: 100 * (s == 0).mean()),
               front_grid_rows=("front_grid", "sum"),
               front_quali_rows=("front_quali", "sum")))
miss["front_grid_without_quali"] = q1_df[q1_df.front_grid].groupby("era")["quali_position"].apply(lambda s: s.isna().sum())
miss.index = miss.index.map(ERA_LABEL)
display(miss.round(2))

# Where does qualifying data actually exist? (it is absent or partial in early seasons)
cov = (q1_df.assign(decade=(q1_df["year"] // 10) * 10)
         .groupby("decade")
         .agg(entries=("podium", "size"), quali_available_pct=("quali_position", lambda s: 100 * s.notna().mean())))
display(cov.round(1))
first_q = int(q1_df.loc[q1_df["quali_position"].notna(), "year"].min())
print("first season with any qualifying position:", first_q)

## 4. Qualifying position vs grid

The two definitions disagree when a penalty or pit-lane start moves a driver. Each front-row entry falls into one of three groups:

- **both**: qualified and started on the front row.
- **qualified only**: qualified in the top 2 but a penalty pushed them off the front row.
- **grid only**: started on the front row despite not qualifying there (promoted when someone ahead was penalised or excluded).

In [ ]:
q1_d = q1_df.dropna(subset=["quali_position"]).copy()
q1_d["group"] = np.select(
    [q1_d.front_grid & q1_d.front_quali, q1_d.front_quali & ~q1_d.front_grid, q1_d.front_grid & ~q1_d.front_quali],
    ["both (quali & grid)", "qualified only (penalised)", "grid only (promoted)"],
    default="neither")
pen = (q1_d[q1_d.group != "neither"].groupby(["era", "group"])
         .agg(n=("podium", "size"), podium_rate=("podium", "mean")).reset_index())
pen["era"] = pen["era"].map(ERA_LABEL)
display(pen.round(3))

## 5. Conversion rate per circuit and era

`conversion rate = podiums / front-row starts`, with a 95% Wilson interval. Circuits need at least `MIN_N` front-row starts in an era to be ranked there. Rank changes are computed **only over circuits ranked in both eras**, so both rankings are over the same set of circuits.

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half

def conversion(frame, flag):
    g = (frame[frame[flag]]
         .groupby(["circuitId", "circuit_name", "era"])
         .agg(n=("podium", "size"), podiums=("podium", "sum"), races=("raceId", "nunique"))
         .reset_index())
    g["n"] = g["n"].astype(float); g["podiums"] = g["podiums"].astype(float)
    g["rate"] = g["podiums"] / g["n"]
    g["ci_lo"], g["ci_hi"] = wilson(g["podiums"], g["n"])
    return g

def compare(conv):
    ok = conv[conv["n"] >= MIN_N]
    wide = ok.pivot(index=["circuitId", "circuit_name"], columns="era", values=["n", "rate", "ci_lo", "ci_hi"])
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()
    both = wide.dropna(subset=["rate_pre", "rate_post"]).copy()
    both["rank_pre"]    = both["rate_pre"].rank(ascending=False, method="min").astype(int)
    both["rank_post"]   = both["rate_post"].rank(ascending=False, method="min").astype(int)
    both["rank_change"] = both["rank_pre"] - both["rank_post"]   # + = climbed after 2014
    both["rate_delta"]  = both["rate_post"] - both["rate_pre"]
    return both.sort_values("rate_post", ascending=False), wide

conv_grid  = conversion(q1_df, "front_grid")
cmp_grid, wide_grid = compare(conv_grid)

conv_quali = conversion(q1_df.dropna(subset=["quali_position"]), "front_quali")
cmp_quali, wide_quali = compare(conv_quali)

dropped = conv_grid.groupby("era").apply(lambda g: (g["n"] < MIN_N).sum())
print("circuit-era cells below MIN_N (not ranked):", dropped.to_dict())
print("circuits ranked in both eras (grid definition):", len(cmp_grid))
display(cmp_grid[["circuit_name", "n_pre", "rate_pre", "rank_pre", "n_post", "rate_post", "rank_post", "rank_change"]].round(3))

## 6. Does the ranking change?

In [ ]:
def overall(frame, flag):
    g = frame[frame[flag]].groupby("era").agg(n=("podium", "size"), podiums=("podium", "sum"))
    g["rate"] = g["podiums"] / g["n"]
    g["ci_lo"], g["ci_hi"] = wilson(g["podiums"], g["n"])
    g.index = g.index.map(ERA_LABEL)
    return g

print("Overall front-row (grid) conversion by era")
display(overall(q1_df, "front_grid").round(3))

rho, p = spearmanr(cmp_grid["rate_pre"], cmp_grid["rate_post"])
print(f"Spearman rank correlation of circuit conversion, pre vs hybrid (grid): rho={rho:.2f}, p={p:.3f}, circuits={len(cmp_grid)}")

# agreement between the two front-row definitions
for era in ["pre", "post"]:
    a = wide_grid.set_index("circuitId")[f"rate_{era}"]
    b = wide_quali.set_index("circuitId")[f"rate_{era}"]
    j = pd.concat([a, b], axis=1, keys=["grid", "quali"]).dropna()
    r_, p_ = spearmanr(j["grid"], j["quali"])
    print(f"grid vs qualifying definition, {ERA_LABEL[era]}: rho={r_:.2f} (circuits={len(j)})")

top = 5
for era in ["pre", "post"]:
    t = wide_grid.dropna(subset=[f"rate_{era}"]).sort_values(f"rate_{era}", ascending=False)
    print(f"\nTop {top} circuits, {ERA_LABEL[era]}:",
          [f"{n} ({r:.0%}, n={int(k)})" for n, r, k in zip(t.circuit_name.head(top), t[f"rate_{era}"].head(top), t[f"n_{era}"].head(top))])
print("\nBiggest climbers:", cmp_grid.nlargest(3, "rank_change")[["circuit_name", "rank_pre", "rank_post"]].values.tolist())
print("Biggest fallers: ", cmp_grid.nsmallest(3, "rank_change")[["circuit_name", "rank_pre", "rank_post"]].values.tolist())

## 7. Visualisations

In [ ]:
def dumbbell(cmp, title):
    c = cmp.sort_values("rate_post")
    y = np.arange(len(c))
    fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(c))))
    ax.hlines(y, c["rate_pre"], c["rate_post"], color="lightgrey", zorder=1)
    ax.scatter(c["rate_pre"],  y, color="#4c78a8", label=ERA_LABEL["pre"],  zorder=2)
    ax.scatter(c["rate_post"], y, color="#e10600", label=ERA_LABEL["post"], zorder=2)
    ax.set_yticks(y); ax.set_yticklabels(c["circuit_name"], fontsize=8)
    ax.set_xlabel("Front-row start → podium rate"); ax.set_xlim(0, 1.02)
    ax.set_title(title, fontweight="bold"); ax.legend(loc="lower right"); ax.grid(axis="x", alpha=.3)
    plt.tight_layout(); plt.show()

def slope(cmp, title):
    fig, ax = plt.subplots(figsize=(7, max(5, 0.34 * len(cmp))))
    for _, r in cmp.iterrows():
        col = "#2ca02c" if r.rank_change > 0 else "#d62728" if r.rank_change < 0 else "grey"
        ax.plot([0, 1], [r.rank_pre, r.rank_post], color=col, marker="o")
        ax.text(-0.03, r.rank_pre,  r.circuit_name, ha="right", va="center", fontsize=7)
        ax.text(1.03,  r.rank_post, r.circuit_name, ha="left",  va="center", fontsize=7)
    ax.set_xlim(-0.8, 1.8); ax.invert_yaxis(); ax.set_xticks([0, 1])
    ax.set_xticklabels([ERA_LABEL["pre"], ERA_LABEL["post"]]); ax.set_ylabel("Rank (1 = highest conversion)")
    ax.set_title(title, fontweight="bold"); ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

dumbbell(cmp_grid, f"Front-row (grid) → podium rate by circuit (n ≥ {MIN_N} per era)")
slope(cmp_grid, "Circuit ranking: pre-hybrid vs hybrid era (grid definition)")
dumbbell(cmp_quali, "Sensitivity: front-row *qualifiers* → podium rate")

## 8. Answer, justification and caveats

> **Write the final answer here after running the cells above.** Use the printed top-5 lists, the Spearman ρ, and the climbers/fallers. Suggested structure:
>
> **Answer.** The circuits with the highest front-row→podium conversion were … (pre-hybrid) and … (hybrid). The ranking *[changed substantially / was largely stable]* (Spearman ρ = …); the biggest movers were … .
>
> **Grain.** One row per (`raceId`, `driverId`) after Rule A dedup; a "front-row start" is one driver-race entry, so each race contributes at most two.
>
> **Missingness.** Qualifying rows missing for …% of entries (see §3); `grid = 0` entries are excluded from the front row; circuit-era cells with fewer than `MIN_N` front-row starts are not ranked, and circuits present in only one era (e.g. new or dropped venues) are excluded from the rank-change comparison.
>
> **Qualifying vs grid.** Primary result uses the actual grid slot; the qualifying-position version agrees with it at ρ = … . Penalised front-row qualifiers convert at …% vs …% for those who kept the slot.
>
> **Pre-race cutoff.** `grid` and `quali_position` are both known before lights-out, so they are valid pre-race inputs. `positionOrder` (podium) is the outcome and is used only as the label here. These per-circuit rates are descriptive and use all races, so if reused as a *model feature* they must be recomputed with an expanding window that only includes races strictly before the one being predicted.
>
> **Early-era caveats.** The full history is used. Before the 1960s to 1970s, grids were often 3-wide (a "front row" was 3 cars) and `grid = 0` was common, so `grid ∈ {1, 2}` is a consistent rule rather than a literal front row. Reliability and car attrition were much higher, which lowers podium conversion in the early pre-hybrid era. The pre-hybrid rank is therefore a long-run average over very different regimes; if desired, add a sub-era split (e.g. 1950–1993, 1994–2013) as a robustness check.
>
> **Caveats.** Small samples per circuit (about 2 front-row starts per race), circuit layout changes under one `circuitId`, and car-dominance effects (e.g. a dominant team's era) confound circuit effects.